# Huấn luyện Ablation DeepVQE C1 trên Kaggle — Phase 1

Notebook này theo cấu trúc của `train_base_deepvqe_colab_v4.ipynb`: setup môi trường, clone repo, tải/tạo CSV, cấu hình, Dataset/DataLoader, khởi tạo model, utility loss/checkpoint, training loop, kiểm tra checkpoint và đánh giá.

Notebook chỉ train và đánh giá riêng `C1 = DeepVQE NS-only + Depthwise Separable Conv trong ResidualBlock`. Không cần baseline/B1a/B1b checkpoint ở đây; notebook so sánh nhiều checkpoint sẽ làm riêng sau.

## 1. Cài đặt môi trường & chuẩn bị Kaggle Working Dir

In [ ]:
!pip install -q wandb torchaudio tensorboard soundfile pandas tqdm einops pesq pystoi pyyaml

import os
import sys
from pathlib import Path

WORK_DIR = Path('/kaggle/working/DeepVQE_Workspace')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

print(f'Thư mục làm việc hiện tại: {Path.cwd()}')
print(f'Python executable: {sys.executable}')

## 2. Clone mã nguồn DeepVQE

In [ ]:
# Repo này cần có deepvqe.py, ablation/ và stream/.
import subprocess

GIT_REPO = 'https://github.com/hoxuanphu/Pdeepvqe.git'
REPO_DIR = WORK_DIR / 'deepvqe'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', GIT_REPO, str(REPO_DIR)], check=True)
else:
    print(f'Thư mục {REPO_DIR} đã tồn tại. Đang cập nhật code mới nhất...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f'Đã vào thư mục mã nguồn: {Path.cwd()}')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=False)

## 3. Tải bộ dữ liệu VoiceBank-DEMAND

Dữ liệu gốc được tải từ Đại học Edinburgh. Nếu bạn đã gắn Kaggle Dataset riêng, chỉ cần sửa `data_dir` trỏ tới thư mục chứa bốn folder wav tương ứng.

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path

# Thư mục chứa file ZIP trên Google Drive
drive_data_dir = Path('/kaggle/working/DeepVQE_Workspace/data/voicebank-demand')
drive_data_dir.mkdir(parents=True, exist_ok=True)

# Thư mục trên máy ảo Kaggle để giải nén và train cho nhanh
data_dir = Path('/kaggle/working/data/voicebank-demand')
data_dir.mkdir(parents=True, exist_ok=True)

datasets = [
    ("clean_trainset_28spk_wav.zip", "https://drive.google.com/file/d/1NJr2O4Ik6ueSFlIGSvub8dnFXGTHJ2PG/view?usp=sharing"),
    ("noisy_trainset_28spk_wav.zip", "https://drive.google.com/file/d/1OqpDIvpVyaTnMbwY1Qt__hfX3X4siMtU/view?usp=sharing"),
    ("clean_testset_wav.zip", "https://drive.google.com/file/d/1GQc-T1R4FNrhRjTn7AAvAenZTIQEazeH/view?usp=sharing"),
    ("noisy_testset_wav.zip", "https://drive.google.com/file/d/1rimmCqxXRYRIXZcPkGjQiacr6j1QsMAH/view?usp=sharing")
]

for filename, url in datasets:
    drive_zip = drive_data_dir / filename
    local_zip = data_dir / filename
    extract_folder = data_dir / filename.replace('.zip', '')
    
    # 1. Ưu tiên kiểm tra trên Google Drive trước
    if drive_zip.exists():
        print(f"✅ Đã tìm thấy {filename} trên Google Drive!")
        if not local_zip.exists() and not extract_folder.exists():
            print(f"Đang copy {filename} từ Drive sang Local SSD...")
            shutil.copy2(drive_zip, local_zip)
    else:
        # 2. Không có trên Drive -> Tải trực tiếp từ web xuống Local SSD
        print(f"🌐 Không thấy {filename} trên Drive, đang tải trực tiếp từ web xuống Local SSD...")
        if not extract_folder.exists():
            if local_zip.exists() and not zipfile.is_zipfile(local_zip):
                print(f"File {local_zip} bị lỗi (không phải ZIP), đang xóa...")
                local_zip.unlink()
            if not local_zip.exists():
                os.system(f"gdown --fuzzy {url} -O {local_zip}")
    
    # 3. Giải nén trên Local SSD
    if not extract_folder.exists():
        if local_zip.exists():
            print(f"Đang giải nén {filename}...")
            with zipfile.ZipFile(local_zip, 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            print(f"Hoàn tất giải nén {filename}.")
        else:
            print(f"❌ Lỗi: Không tìm thấy file {local_zip} để giải nén!")

## 4. Xử lý và phân chia Dataset (Tạo file CSV)

Tạo `train.csv`, `valid.csv`, `test.csv` giống baseline v4: train/valid split 90/10 với `random_state=42`.

In [ ]:
import glob
import pandas as pd
from pathlib import Path

data_dir = Path(data_dir)


def create_csv(clean_dir, noisy_dir, output_csv):
    clean_files = sorted(glob.glob(str(Path(clean_dir) / '*.wav')))
    noisy_files = sorted(glob.glob(str(Path(noisy_dir) / '*.wav')))

    assert len(clean_files) == len(noisy_files), f'Số lượng file không khớp! ({len(clean_files)} != {len(noisy_files)})'
    mismatched = [
        (Path(c).name, Path(n).name)
        for c, n in zip(clean_files, noisy_files)
        if Path(c).name != Path(n).name
    ]
    assert not mismatched, f'Tên file clean/noisy không khớp: {mismatched[:5]}'

    rows = []
    for clean, noisy in zip(clean_files, noisy_files):
        filename = Path(clean).name
        rows.append({
            'ID': filename.replace('.wav', ''),
            'clean_wav': str(Path(clean).resolve()),
            'noisy_wav': str(Path(noisy).resolve()),
        })

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f'Đã tạo {output_csv} với {len(df)} mẫu.')


create_csv(data_dir / 'clean_trainset_28spk_wav', data_dir / 'noisy_trainset_28spk_wav', data_dir / 'train_full.csv')
create_csv(data_dir / 'clean_testset_wav', data_dir / 'noisy_testset_wav', data_dir / 'test.csv')

train_full = pd.read_csv(data_dir / 'train_full.csv')
train_full = train_full.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(train_full) * 0.90)
train_full.iloc[:split_idx].to_csv(data_dir / 'train.csv', index=False)
train_full.iloc[split_idx:].to_csv(data_dir / 'valid.csv', index=False)

print(f'Train: {split_idx} | Valid: {len(train_full) - split_idx}')
print(f'Test: {len(pd.read_csv(data_dir / "test.csv"))}')
print('Hoàn tất tạo metadata CSV!')

## 5. Cấu hình Hyperparameters

Các giá trị bám baseline v4: seed `1234`, STFT `sqrt_hann`, batch size `8`, optimizer Adam, loss RI/Mag/SI-SNR. Notebook chỉ train `C1`.

In [ ]:
import json
from pathlib import Path

data_dir = Path(data_dir)
WORK_DIR = Path(WORK_DIR)

CONFIG = {
    'config_id': 'C1',
    'seed': 1234,
    'sample_rate': 16000,
    'n_fft': 512,
    'hop_length': 256,
    'win_length': 512,
    'stft_window': 'sqrt_hann',

    'batch_size': 8,
    'valid_batch_size': 4,
    'grad_accum_steps': 1,
    'num_workers': 2,
    'persistent_workers': True,
    'prefetch_factor': 2,
    'progress_update_every': 10,
    'epochs': 80,
    'lr': 1e-3,
    'weight_decay': 0.0,
    'grad_clip': 5.0,

    'lamda_ri': 30.0,
    'lamda_mag': 70.0,
    'compress_factor': 0.3,

    'train_csv': str(data_dir / 'train.csv'),
    'valid_csv': str(data_dir / 'valid.csv'),
    'test_csv': str(data_dir / 'test.csv'),

    'checkpoint_dir': str(WORK_DIR / 'runs' / 'ablation' / 'C1'),
    'output_dir': str(WORK_DIR / 'runs' / 'ablation' / 'C1'),
    'resume_checkpoint': 'last.pt',
    'results_dir': str(WORK_DIR / 'results' / 'ablation'),
    'onnx_dir': str(WORK_DIR / 'onnx_models' / 'ablation'),

    'scheduler_factor': 0.5,
    'scheduler_patience': 5,
    'scheduler_min_lr': 1e-6,
    'early_stopping_patience': 15,
    'early_stopping_min_delta': 0.0,
    'early_stopping_min_epochs': 0,

    'use_amp': True,
    'augment': True,
    'aug_gain_range_db': (-6.0, 6.0),
    'aug_snr_remix_range': (0.0, 20.0),
    'aug_prob': 0.5,

    # True để khớp baseline v4 hiện tại. Khi chốt phương án và train sạch, đổi thành False.
    'valid_random_crop': True,

    'tensorboard': False,
    'use_wandb': False,
    'wandb_project': 'DeepVQE-Ablation',
    'eval_pesq_every': 5,
    'eval_pesq_samples': 50,
    'log_audio_every': 0,

    'run_training': True,
    'run_smoke_tests': True,
    'run_arch_benchmark': True,
    'run_quality_eval': True,
    'run_onnx_export': False,
}

for key in ['checkpoint_dir', 'output_dir', 'results_dir', 'onnx_dir']:
    Path(CONFIG[key]).mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, indent=2, default=str))

## 6. Dataset & DataLoader (Pure PyTorch)

Cùng kiểu baseline v4: crop đoạn 3 giây để tiết kiệm VRAM, augment chỉ áp dụng cho training set.

In [ ]:
import random
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CONFIG['seed'])


def repeat_to_length(wav, length):
    if wav.numel() == 0:
        raise ValueError('Encountered empty waveform')
    if wav.numel() >= length:
        return wav
    repeats = int(np.ceil(length / wav.numel()))
    return wav.repeat(repeats)[:length]


def crop_pair(noisy, clean, segment_len, random_crop=True):
    min_len = min(noisy.shape[0], clean.shape[0])
    noisy = noisy[:min_len]
    clean = clean[:min_len]

    if segment_len is None:
        return noisy, clean
    if min_len <= segment_len:
        return noisy, clean

    if random_crop:
        start = random.randint(0, min_len - segment_len)
    else:
        start = max(0, (min_len - segment_len) // 2)
    return noisy[start:start + segment_len], clean[start:start + segment_len]


class VCTKDemandDataset(Dataset):
    """Dataset cho VCTK-DEMAND: đọc cặp (noisy, clean) từ CSV."""

    def __init__(self, csv_path, sample_rate=16000, segment_len=None, augment_cfg=None, random_crop=True):
        self.df = pd.read_csv(csv_path)
        self.sample_rate = sample_rate
        self.segment_len = segment_len
        self.aug = augment_cfg
        self.random_crop = random_crop

    def __len__(self):
        return len(self.df)

    def _load(self, path):
        wav, sr = torchaudio.load(path)
        wav = wav.float()
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
        return wav.squeeze(0)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        noisy = self._load(row['noisy_wav'])
        clean = self._load(row['clean_wav'])
        noisy, clean = crop_pair(noisy, clean, self.segment_len, random_crop=self.random_crop)

        if self.aug is not None:
            aug_prob = self.aug.get('aug_prob', 0.5)

            if random.random() < aug_prob:
                lo, hi = self.aug['aug_gain_range_db']
                gain_db = random.uniform(lo, hi)
                gain = 10 ** (gain_db / 20.0)
                noisy = noisy * gain
                clean = clean * gain

            if random.random() < aug_prob:
                noise = noisy - clean
                lo, hi = self.aug['aug_snr_remix_range']
                target_snr = random.uniform(lo, hi)
                clean_rms = clean.pow(2).mean().sqrt().clamp(min=1e-8)
                noise_rms = noise.pow(2).mean().sqrt().clamp(min=1e-8)
                target_noise_rms = clean_rms / (10 ** (target_snr / 20.0))
                noisy = clean + noise * (target_noise_rms / noise_rms)

        return {'noisy': noisy, 'clean': clean, 'id': row['ID']}


def collate_fn(batch):
    max_len = max(item['noisy'].shape[0] for item in batch)
    noisy_batch, clean_batch, ids = [], [], []
    for item in batch:
        noisy, clean = item['noisy'], item['clean']
        pad_len = max_len - noisy.shape[0]
        if pad_len > 0:
            noisy = torch.nn.functional.pad(noisy, (0, pad_len))
            clean = torch.nn.functional.pad(clean, (0, pad_len))
        noisy_batch.append(noisy)
        clean_batch.append(clean)
        ids.append(item['id'])

    return {
        'noisy': torch.stack(noisy_batch),
        'clean': torch.stack(clean_batch),
        'id': ids,
    }


segment_len = int(3.0 * CONFIG['sample_rate'])
aug_cfg = {k: v for k, v in CONFIG.items() if k.startswith('aug_')} if CONFIG.get('augment') else None

train_dataset = VCTKDemandDataset(
    CONFIG['train_csv'],
    CONFIG['sample_rate'],
    segment_len=segment_len,
    augment_cfg=aug_cfg,
    random_crop=True,
)
valid_dataset = VCTKDemandDataset(
    CONFIG['valid_csv'],
    CONFIG['sample_rate'],
    segment_len=segment_len,
    augment_cfg=None,
    random_crop=CONFIG.get('valid_random_crop', True),
)

loader_kwargs = dict(
    num_workers=CONFIG['num_workers'],
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
if CONFIG['num_workers'] > 0:
    loader_kwargs.update(
        persistent_workers=CONFIG.get('persistent_workers', False),
        prefetch_factor=CONFIG.get('prefetch_factor', 2),
    )

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=CONFIG.get('valid_batch_size', CONFIG['batch_size']),
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Valid: {len(valid_dataset)} samples, {len(valid_loader)} batches')
print(f'Valid crop: {"random như baseline v4" if CONFIG.get("valid_random_crop", True) else "center/fixed"}')
if aug_cfg:
    print(f'Augmentation: ON (gain={aug_cfg["aug_gain_range_db"]}, snr_remix={aug_cfg["aug_snr_remix_range"]}, prob={aug_cfg["aug_prob"]})')
else:
    print('Augmentation: OFF')

## 7. Khởi tạo Model C1, Optimizer, Scheduler

Khác baseline gốc ở đúng một điểm: model dùng `DeepVQE_Ablation.from_config_id('C1')`.

In [ ]:
import time
import json
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

from ablation.ablation_config import deep_update, get_train_config, reproducibility_metadata
from ablation.deepvqe_ablation import DeepVQE_Ablation, count_parameters


def make_grad_scaler(enabled):
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        return torch.amp.GradScaler('cuda', enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast_context(device, enabled):
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
        return torch.amp.autocast(device.type, enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)


def build_train_cfg(config):
    override = {
        'experiment': {
            'config_id': config['config_id'],
            'name': f"deepvqe_ablation_{config['config_id']}",
            'seed': config['seed'],
            'output_dir': config['output_dir'],
            'resume_from': None,
        },
        'stft': {
            'n_fft': config['n_fft'],
            'hop_length': config['hop_length'],
            'win_length': config['win_length'],
            'window': config['stft_window'],
        },
        'data': {
            'sample_rate': config['sample_rate'],
            'clip_seconds': 3.0,
            'num_workers': config['num_workers'],
            'pin_memory': torch.cuda.is_available(),
            'train_manifest': config['train_csv'],
            'valid_manifest': config['valid_csv'],
            'test_manifest': config['test_csv'],
            'augment': config['augment'],
            'aug_gain_range_db': list(config['aug_gain_range_db']),
            'aug_snr_remix_range': list(config['aug_snr_remix_range']),
            'aug_prob': config['aug_prob'],
        },
        'optimizer': {
            'lr': config['lr'],
            'weight_decay': config['weight_decay'],
            'betas': [0.9, 0.999],
            'grad_clip_norm': config['grad_clip'],
        },
        'scheduler': {
            'mode': 'min',
            'factor': config['scheduler_factor'],
            'patience': config['scheduler_patience'],
            'min_lr': config['scheduler_min_lr'],
        },
        'training': {
            'device': 'cuda' if torch.cuda.is_available() else 'cpu',
            'batch_size': config['batch_size'],
            'valid_batch_size': config['valid_batch_size'],
            'epochs': config['epochs'],
            'grad_accum_steps': config['grad_accum_steps'],
            'checkpoint_monitor': 'loss',
            'checkpoint_mode': 'min',
            'use_amp': config['use_amp'],
            'early_stop_patience': config['early_stopping_patience'],
            'early_stop_min_delta': config['early_stopping_min_delta'],
            'early_stop_min_epochs': config['early_stopping_min_epochs'],
            'valid_random_crop': config['valid_random_crop'],
        },
        'loss': {
            'lamda_ri': config['lamda_ri'],
            'lamda_mag': config['lamda_mag'],
            'compress_factor': config['compress_factor'],
        },
        'logging': {
            'use_wandb': config['use_wandb'],
            'wandb_project': config['wandb_project'],
            'eval_pesq_every': config['eval_pesq_every'],
            'eval_pesq_samples': config['eval_pesq_samples'],
            'log_audio_every': config['log_audio_every'],
        },
    }
    return deep_update(get_train_config(config['config_id']), override)


TRAIN_CFG = build_train_cfg(CONFIG)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

model = DeepVQE_Ablation.from_config_id(CONFIG['config_id']).to(device)
USE_DATA_PARALLEL = False
if USE_DATA_PARALLEL and torch.cuda.device_count() > 1:
    print(f'Bật DataParallel trên {torch.cuda.device_count()} GPUs')
    model = torch.nn.DataParallel(model)
else:
    print('DataParallel: OFF')

total_params = count_parameters(model.module if hasattr(model, 'module') else model)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params / 1e6:.2f}M | Trainable: {trainable_params / 1e6:.2f}M')

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=CONFIG['scheduler_factor'],
    patience=CONFIG['scheduler_patience'],
    min_lr=CONFIG['scheduler_min_lr'],
)

use_amp = CONFIG['use_amp'] and device.type == 'cuda'
scaler = make_grad_scaler(enabled=use_amp)
print(f'Mixed Precision (AMP): {"ON" if use_amp else "OFF"}')

window = torch.hann_window(CONFIG['win_length']).sqrt().to(device)

output_dir = Path(CONFIG.get('checkpoint_dir', CONFIG['output_dir']))
output_dir.mkdir(parents=True, exist_ok=True)
Path(CONFIG['results_dir']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['onnx_dir']).mkdir(parents=True, exist_ok=True)

with open(output_dir / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(TRAIN_CFG, f, indent=2, default=str)
with open(output_dir / 'notebook_config.json', 'w', encoding='utf-8') as f:
    json.dump(CONFIG, f, indent=2, default=str)

tb_log_dir = output_dir / 'tb_logs'
tb_writer = SummaryWriter(log_dir=str(tb_log_dir)) if CONFIG.get('tensorboard', False) else None
print(f'Output dir: {output_dir}')
print(f'TensorBoard: {"ON" if tb_writer else "OFF"}')

## 8. Hàm tiện ích: STFT, Loss, Checkpoint, Log, Metrics

Giữ cùng loss baseline v4: compressed RI + compressed magnitude + negative SI-SNR trên waveform gốc.

In [ ]:
import csv
import shutil


def make_stft(wav, n_fft, hop_length, win_length, win):
    spec = torch.stft(
        wav,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=win,
        return_complex=True,
    )
    return torch.view_as_real(spec)


def make_istft(spec, n_fft, hop_length, win_length, win, length=None):
    complex_spec = torch.complex(spec[..., 0], spec[..., 1])
    return torch.istft(
        complex_spec,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=win,
        length=length,
    )


def compute_loss(model, noisy_wav, clean_wav, cfg, win):
    n_fft = cfg['n_fft']
    hop = cfg['hop_length']
    win_len = cfg['win_length']
    c = cfg['compress_factor']

    noisy_spec = make_stft(noisy_wav, n_fft, hop, win_len, win)

    amp_enabled = cfg.get('use_amp', False) and noisy_wav.device.type == 'cuda'
    with autocast_context(noisy_wav.device, enabled=amp_enabled):
        pred_stft = model(noisy_spec)

    pred_stft = pred_stft.float()
    clean_spec = make_stft(clean_wav, n_fft, hop, win_len, win)

    min_t = min(pred_stft.shape[2], clean_spec.shape[2])
    pred_stft = pred_stft[:, :, :min_t, :]
    true_stft = clean_spec[:, :, :min_t, :]

    pred_real, pred_imag = pred_stft[..., 0], pred_stft[..., 1]
    true_real, true_imag = true_stft[..., 0], true_stft[..., 1]

    pred_mag = torch.sqrt(pred_real**2 + pred_imag**2 + 1e-12)
    true_mag = torch.sqrt(true_real**2 + true_imag**2 + 1e-12)

    pred_real_c = pred_real / (pred_mag**(1 - c))
    pred_imag_c = pred_imag / (pred_mag**(1 - c))
    true_real_c = true_real / (true_mag**(1 - c))
    true_imag_c = true_imag / (true_mag**(1 - c))

    real_loss = torch.mean((pred_real_c - true_real_c)**2)
    imag_loss = torch.mean((pred_imag_c - true_imag_c)**2)
    mag_loss = torch.mean((pred_mag**c - true_mag**c)**2)

    y_pred = torch.istft(
        pred_real + 1j * pred_imag,
        n_fft=n_fft,
        hop_length=hop,
        win_length=win_len,
        window=win,
    )
    y_true = clean_wav
    min_wav_len = min(y_pred.shape[-1], y_true.shape[-1])
    y_pred = y_pred[..., :min_wav_len]
    y_true = y_true[..., :min_wav_len]

    eps = 1e-8
    true_energy = torch.sum(torch.square(y_true), dim=-1, keepdim=True)
    y_target = torch.sum(y_true * y_pred, dim=-1, keepdim=True) * y_true / (true_energy + eps)
    target_energy = torch.sum(torch.square(y_target), dim=-1, keepdim=True)
    noise_energy = torch.sum(torch.square(y_pred - y_target), dim=-1, keepdim=True)
    sisnr = -10 * torch.log10((target_energy + eps) / (noise_energy + eps)).mean()

    loss = cfg['lamda_ri'] * (real_loss + imag_loss) + cfg['lamda_mag'] * mag_loss + sisnr
    return loss, {
        'loss': loss.item(),
        'ri_loss': (real_loss + imag_loss).item(),
        'mag_loss': mag_loss.item(),
        'sisnr': sisnr.item(),
    }


def unwrap_model(model):
    return model.module if hasattr(model, 'module') else model


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    model_state = unwrap_model(model).state_dict()
    metadata = reproducibility_metadata(TRAIN_CFG, checkpoint_id=path.stem)
    ckpt = {
        'model': model_state,
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler is not None else None,
        'scaler': scaler.state_dict() if scaler is not None else None,
        'epoch': epoch,
        'best_loss': best_loss,
        'best_metric': best_loss,
        'bad_epochs': bad_epochs,
        'config': TRAIN_CFG,
        'metadata': metadata,
    }

    temp_local = Path('/kaggle/working') / f'temp_{path.name}' if Path('/kaggle/working').exists() else Path(f'temp_{path.name}')
    temp_target = path.with_suffix(path.suffix + '.tmp')
    torch.save(ckpt, temp_local)
    shutil.copy2(temp_local, temp_target)
    os.replace(temp_target, path)
    if temp_local.exists():
        temp_local.unlink()


def load_checkpoint(path, model, optimizer=None, scheduler=None, device='cpu', scaler=None):
    ckpt = torch.load(str(path), map_location=device, weights_only=False)
    unwrap_model(model).load_state_dict(ckpt['model'])
    if optimizer is not None and ckpt.get('optimizer'):
        optimizer.load_state_dict(ckpt['optimizer'])
    if scheduler is not None and ckpt.get('scheduler'):
        scheduler.load_state_dict(ckpt['scheduler'])
    if scaler is not None and ckpt.get('scaler'):
        scaler.load_state_dict(ckpt['scaler'])
    return ckpt.get('epoch', 0), ckpt.get('best_loss', ckpt.get('best_metric')), ckpt.get('bad_epochs', 0)


def set_optimizer_lr(optimizer, lr):
    for group in optimizer.param_groups:
        group['lr'] = lr


def optimizer_step_with_amp(optimizer, scaler, model, grad_clip):
    if grad_clip:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)


def append_log(log_path, row_dict):
    log_path = Path(log_path)
    file_exists = log_path.exists() and log_path.stat().st_size > 0
    with open(log_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(row_dict.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)


def compute_pesq_stoi(model, dataloader, cfg, win, device, max_samples=50):
    from pesq import pesq
    from pystoi import stoi

    m = unwrap_model(model)
    was_training = m.training
    m.eval()
    sr = cfg['sample_rate']
    pesq_scores, stoi_scores = [], []
    count = 0
    amp_enabled = cfg.get('use_amp', False) and device.type == 'cuda'

    with torch.no_grad():
        for batch in dataloader:
            noisy = batch['noisy'].to(device)
            clean = batch['clean']
            spec = make_stft(noisy, cfg['n_fft'], cfg['hop_length'], cfg['win_length'], win)
            with autocast_context(device, enabled=amp_enabled):
                pred = model(spec)
            enhanced = make_istft(pred.float(), cfg['n_fft'], cfg['hop_length'], cfg['win_length'], win, length=noisy.shape[-1])

            for i in range(enhanced.shape[0]):
                enh_np = enhanced[i].detach().cpu().numpy()
                cln_np = clean[i].numpy()
                min_l = min(len(enh_np), len(cln_np))
                enh_np, cln_np = enh_np[:min_l], cln_np[:min_l]
                try:
                    pesq_scores.append(pesq(sr, cln_np, enh_np, 'wb'))
                except Exception:
                    pass
                try:
                    stoi_scores.append(stoi(cln_np, enh_np, sr, extended=False))
                except Exception:
                    pass
                count += 1
                if count >= max_samples:
                    break
            if count >= max_samples:
                break

    m.train(was_training)
    return {
        'pesq': float(np.mean(pesq_scores)) if pesq_scores else 0.0,
        'stoi': float(np.mean(stoi_scores)) if stoi_scores else 0.0,
    }


print('Hàm tiện ích đã sẵn sàng.')

## 8.5 Khởi động TensorBoard (tùy chọn)

In [ ]:
if CONFIG.get('tensorboard', False):
    ip = get_ipython()
    ip.run_line_magic('load_ext', 'tensorboard')
    ip.run_line_magic('tensorboard', f'--logdir {str(output_dir / "tb_logs")}')
else:
    print('TensorBoard đang tắt trong CONFIG để ưu tiên tốc độ train.')

## 8.6 Đăng nhập WandB (tùy chọn)

In [ ]:
if CONFIG.get('use_wandb', False):
    import os
    import wandb

    wandb_key = os.environ.get('WANDB_API_KEY')
    if not wandb_key:
        try:
            from kaggle_secrets import UserSecretsClient
            wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
        except Exception as exc:
            print(f'Không lấy được WANDB_API_KEY từ Kaggle Secrets/env: {exc}')

    if wandb_key:
        wandb.login(key=wandb_key, relogin=True)
        print('Đã đăng nhập WandB bằng WANDB_API_KEY.')
    else:
        print('Chưa có WANDB_API_KEY; wandb.init có thể yêu cầu đăng nhập thủ công.')
else:
    print('WandB đang tắt trong CONFIG. Đổi use_wandb=True nếu muốn log lên WandB.')

## 9. Vòng lặp huấn luyện C1 (Training Loop)

Auto-resume từ `last.pt` nếu có. Checkpoint tốt nhất được chọn theo `valid_loss`, giống baseline v4.

In [ ]:
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

wandb = None
if CONFIG.get('use_wandb', False):
    import wandb as _wandb
    wandb = _wandb
    wandb.init(project=CONFIG.get('wandb_project', 'DeepVQE-Ablation'), config=CONFIG, resume='allow')
    wandb.watch(unwrap_model(model), log='gradients', log_freq=100)

start_epoch = 0
best_loss = None
bad_epochs = 0

resume_path = output_dir / CONFIG.get('resume_checkpoint', 'last.pt')
if resume_path.exists():
    start_epoch, best_loss, bad_epochs = load_checkpoint(
        resume_path,
        model,
        optimizer,
        scheduler,
        device=device,
        scaler=scaler,
    )
    resumed_lr = optimizer.param_groups[0]['lr']
    effective_lr = min(resumed_lr, CONFIG['lr'])
    set_optimizer_lr(optimizer, effective_lr)
    _best_str = f'{best_loss:.6f}' if best_loss is not None else 'N/A'
    print(f'Đã resume từ epoch {start_epoch}, best_loss={_best_str}, bad_epochs={bad_epochs}')
    print(f'LR checkpoint: {resumed_lr:.2e} | LR config: {CONFIG["lr"]:.2e} -> LR dùng: {effective_lr:.2e}')
else:
    print('Bắt đầu training từ đầu.')
    print(f'LR config: {optimizer.param_groups[0]["lr"]:.2e}')

log_path = output_dir / 'train_log.csv'
print(f'Log file: {log_path}')
print(f'AMP: {"ON" if use_amp else "OFF"} | Batch: {CONFIG["batch_size"]} | Accum: {CONFIG["grad_accum_steps"]}')

if CONFIG.get('run_training', True):
    accum_steps = CONFIG.get('grad_accum_steps', 1)
    progress_update_every = max(1, CONFIG.get('progress_update_every', 10))

    for epoch in range(start_epoch + 1, CONFIG['epochs'] + 1):
        model.train()
        train_losses = []
        skipped_train_batches = 0
        valid_accum_batches = 0
        t0 = time.time()
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(train_loader, desc=f'Epoch {epoch} [Train]', leave=False)
        for batch_idx, batch in enumerate(pbar):
            noisy = batch['noisy'].to(device, non_blocking=True)
            clean = batch['clean'].to(device, non_blocking=True)

            loss, parts = compute_loss(model, noisy, clean, CONFIG, window)
            if not torch.isfinite(loss):
                skipped_train_batches += 1
                optimizer.zero_grad(set_to_none=True)
                print(f'  [WARN] Skip train batch {batch_idx}: non-finite loss | ids={batch["id"][:3]}')
                continue

            scaler.scale(loss / accum_steps).backward()
            valid_accum_batches += 1
            if valid_accum_batches % accum_steps == 0:
                optimizer_step_with_amp(optimizer, scaler, model, CONFIG['grad_clip'])

            train_losses.append(parts)
            if batch_idx % progress_update_every == 0 or (batch_idx + 1) == len(train_loader):
                pbar.set_postfix({'loss': f"{parts['loss']:.4f}"})

        if valid_accum_batches % accum_steps != 0:
            optimizer_step_with_amp(optimizer, scaler, model, CONFIG['grad_clip'])
        if not train_losses:
            print('Không có train batch hợp lệ trong epoch này; dừng training để tránh ghi checkpoint lỗi.')
            break
        avg_train = {k: float(np.mean([d[k] for d in train_losses])) for k in train_losses[0]}
        elapsed = time.time() - t0

        model.eval()
        valid_losses = []
        skipped_valid_batches = 0
        with torch.no_grad():
            for batch in tqdm(valid_loader, desc=f'Epoch {epoch} [Valid]', leave=False):
                noisy = batch['noisy'].to(device, non_blocking=True)
                clean = batch['clean'].to(device, non_blocking=True)
                valid_loss, parts = compute_loss(model, noisy, clean, CONFIG, window)
                if not torch.isfinite(valid_loss):
                    skipped_valid_batches += 1
                    print(f'  [WARN] Skip valid batch: non-finite loss | ids={batch["id"][:3]}')
                    continue
                valid_losses.append(parts)

        if not valid_losses:
            print('Không có valid batch hợp lệ trong epoch này; dừng training để tránh ghi checkpoint lỗi.')
            break
        avg_valid = {k: float(np.mean([d[k] for d in valid_losses])) for k in valid_losses[0]}
        train_has_nan = skipped_train_batches > 0 or not np.isfinite(avg_train['loss'])
        valid_has_nan = skipped_valid_batches > 0 or not np.isfinite(avg_valid['loss'])

        pesq_stoi = {'pesq': 0.0, 'stoi': 0.0}
        eval_every = CONFIG.get('eval_pesq_every', 0)
        if eval_every > 0 and epoch % eval_every == 0:
            print(f'  Đang tính PESQ/STOI trên valid subset (mỗi {eval_every} epoch)...')
            pesq_stoi = compute_pesq_stoi(
                model,
                valid_loader,
                CONFIG,
                window,
                device,
                max_samples=CONFIG.get('eval_pesq_samples', 50),
            )
            print(f'  >> PESQ: {pesq_stoi["pesq"]:.3f} | STOI: {pesq_stoi["stoi"]:.4f}')

        if train_has_nan or valid_has_nan:
            print(f'  [WARN] Epoch {epoch} có batch non-finite: train_skipped={skipped_train_batches}, valid_skipped={skipped_valid_batches}.')
            print('  [WARN] Bỏ qua scheduler/checkpoint epoch này. Training vẫn tiếp tục...')
            continue

        prev_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_valid['loss'])
        current_lr = optimizer.param_groups[0]['lr']
        scheduler_bad_epochs = getattr(scheduler, 'num_bad_epochs', None)
        if current_lr < prev_lr:
            print(f'  >>> Scheduler giảm LR: {prev_lr:.2e} -> {current_lr:.2e}')

        previous_best = best_loss
        is_best = previous_best is None or avg_valid['loss'] < previous_best
        improved_enough = previous_best is None or avg_valid['loss'] < previous_best - CONFIG.get('early_stopping_min_delta', 0.0)
        if is_best:
            best_loss = avg_valid['loss']
        if improved_enough:
            bad_epochs = 0
        else:
            bad_epochs += 1

        save_checkpoint(output_dir / 'last.pt', model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler=scaler)
        if is_best:
            save_checkpoint(output_dir / 'best.pt', model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler=scaler)
            print(f'  >>> Saved best model (valid_loss={best_loss:.6f})')

        print(
            f"Epoch {epoch:3d}/{CONFIG['epochs']} | "
            f"Train Loss: {avg_train['loss']:.6f} (ri={avg_train['ri_loss']:.4f}, mag={avg_train['mag_loss']:.4f}, sisnr={avg_train['sisnr']:.4f}) | "
            f"Valid Loss: {avg_valid['loss']:.6f} (ri={avg_valid['ri_loss']:.4f}, mag={avg_valid['mag_loss']:.4f}, sisnr={avg_valid['sisnr']:.4f}) | "
            f"LR: {current_lr:.2e} | Sched bad: {scheduler_bad_epochs}/{CONFIG['scheduler_patience']} | Time: {elapsed:.0f}s"
        )

        append_log(log_path, {
            'epoch': epoch,
            'train_loss': f"{avg_train['loss']:.6f}",
            'train_ri_loss': f"{avg_train['ri_loss']:.6f}",
            'train_mag_loss': f"{avg_train['mag_loss']:.6f}",
            'train_sisnr': f"{avg_train['sisnr']:.6f}",
            'valid_loss': f"{avg_valid['loss']:.6f}",
            'valid_ri_loss': f"{avg_valid['ri_loss']:.6f}",
            'valid_mag_loss': f"{avg_valid['mag_loss']:.6f}",
            'valid_sisnr': f"{avg_valid['sisnr']:.6f}",
            'valid_pesq': f"{pesq_stoi['pesq']:.4f}",
            'valid_stoi': f"{pesq_stoi['stoi']:.4f}",
            'lr': f'{current_lr:.2e}',
            'scheduler_bad_epochs': scheduler_bad_epochs,
            'time_s': f'{elapsed:.0f}',
        })

        if tb_writer:
            tb_writer.add_scalars('Loss/Total', {'train': avg_train['loss'], 'valid': avg_valid['loss']}, epoch)
            tb_writer.add_scalars('Loss/RI', {'train': avg_train['ri_loss'], 'valid': avg_valid['ri_loss']}, epoch)
            tb_writer.add_scalars('Loss/Mag', {'train': avg_train['mag_loss'], 'valid': avg_valid['mag_loss']}, epoch)
            tb_writer.add_scalars('Loss/SISNR', {'train': avg_train['sisnr'], 'valid': avg_valid['sisnr']}, epoch)
            tb_writer.add_scalar('LR', current_lr, epoch)
            if pesq_stoi['pesq'] > 0:
                tb_writer.add_scalar('Metrics/PESQ', pesq_stoi['pesq'], epoch)
                tb_writer.add_scalar('Metrics/STOI', pesq_stoi['stoi'], epoch)
            tb_writer.flush()

        if wandb is not None and wandb.run is not None:
            wandb_metrics = {
                'epoch': epoch,
                'train/loss': avg_train['loss'],
                'train/ri_loss': avg_train['ri_loss'],
                'train/mag_loss': avg_train['mag_loss'],
                'train/sisnr': avg_train['sisnr'],
                'valid/loss': avg_valid['loss'],
                'valid/ri_loss': avg_valid['ri_loss'],
                'valid/mag_loss': avg_valid['mag_loss'],
                'valid/sisnr': avg_valid['sisnr'],
                'lr': current_lr,
            }
            if pesq_stoi.get('pesq', 0) > 0:
                wandb_metrics['metrics/pesq'] = pesq_stoi['pesq']
                wandb_metrics['metrics/stoi'] = pesq_stoi['stoi']
            wandb.log(wandb_metrics, step=epoch)

        if epoch >= CONFIG.get('early_stopping_min_epochs', 0) and bad_epochs >= CONFIG.get('early_stopping_patience', 15):
            print(f'Early stopping: valid_loss không cải thiện đủ trong {bad_epochs} epoch. best_loss={best_loss:.6f}')
            break

    if tb_writer:
        tb_writer.close()
    if wandb is not None and wandb.run is not None:
        wandb.finish()

    print('\n=== Huấn luyện C1 hoàn tất! ===')
    if best_loss is not None:
        print(f'Best valid loss: {best_loss:.6f}')
    else:
        print('Best valid loss: N/A')
    print(f'Checkpoint: {output_dir}')
    if tb_writer:
        print(f'TensorBoard log: {tb_log_dir}')
else:
    print('run_training=False, bỏ qua training loop.')

## 10. Kiểm tra nhanh sau training

Liệt kê checkpoint/log và nghe thử một mẫu từ validation set.

In [ ]:
import IPython.display as ipd

for name in ['best.pt', 'last.pt', 'config.json', 'notebook_config.json', 'train_log.csv']:
    p = output_dir / name
    if p.exists():
        print(f'{name}: {p} ({p.stat().st_size / 1024:.1f} KB)')
    else:
        print(f'MISSING: {p}')

if (output_dir / 'train_log.csv').exists():
    display(pd.read_csv(output_dir / 'train_log.csv').tail())

best_ckpt_path = output_dir / 'best.pt'
if best_ckpt_path.exists():
    print(f'Tải best checkpoint: {best_ckpt_path}')
    load_checkpoint(best_ckpt_path, model, device=device)

    model.eval()
    with torch.no_grad():
        batch = next(iter(valid_loader))
        noisy = batch['noisy'].to(device)
        clean = batch['clean'].to(device)
        spec = make_stft(noisy[:1], CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window)
        with autocast_context(device, enabled=use_amp):
            pred = model(spec)
        enhanced = make_istft(pred.float(), CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window, length=noisy[:1].shape[-1])

    sr = CONFIG['sample_rate']
    print('Noisy')
    display(ipd.Audio(noisy[0].detach().cpu().numpy(), rate=sr))
    print('Enhanced C1')
    display(ipd.Audio(enhanced[0].detach().cpu().numpy(), rate=sr))
    print('Clean')
    display(ipd.Audio(clean[0].detach().cpu().numpy(), rate=sr))
else:
    print('Chưa có best.pt để nghe thử.')

## 11. Smoke test và Architecture Benchmark cho C1

Chỉ chạy `C1`, không đụng tới baseline/B1a/B1b checkpoint.

In [ ]:
import subprocess
import sys


def run_py(args, cwd=Path.cwd(), check=True):
    cmd = [sys.executable, *[str(arg) for arg in args]]
    print('\n$ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd), text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f'Lệnh lỗi với exit code {result.returncode}: {cmd}')
    return result


AB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
arch_csv = Path(CONFIG['results_dir']) / 'ablation_arch_benchmark.csv'

if CONFIG.get('run_smoke_tests', True):
    run_py(['ablation/test_ablation_streaming.py', '--configs', CONFIG['config_id']])
else:
    print('run_smoke_tests=False, bỏ qua smoke test.')

if CONFIG.get('run_arch_benchmark', True):
    run_py([
        'ablation/run_ablation_benchmark.py',
        '--output', arch_csv,
        '--configs', CONFIG['config_id'],
        '--device', AB_DEVICE,
        '--frames', '63',
        '--warmup', '1',
        '--repeats', '3',
    ])
else:
    print('run_arch_benchmark=False, bỏ qua architecture benchmark.')

if arch_csv.exists():
    display(pd.read_csv(arch_csv))

## 12. Đánh giá chất lượng C1: PESQ, STOI, SI-SDR

Chỉ đánh giá `C1/best.pt` trên fixed `test.csv`. So sánh nhiều checkpoint sẽ làm ở notebook/script riêng.

In [ ]:
quality_csv = Path(CONFIG['results_dir']) / 'ablation_quality.csv'
best_ckpt = output_dir / 'best.pt'

if CONFIG.get('run_quality_eval', True):
    if not best_ckpt.exists():
        raise FileNotFoundError(f'Không tìm thấy checkpoint để eval: {best_ckpt}')
    run_py([
        'ablation/eval_ablation_quality.py',
        '--config-id', CONFIG['config_id'],
        '--checkpoint', best_ckpt,
        '--manifest', CONFIG['test_csv'],
        '--output', quality_csv,
        '--device', AB_DEVICE,
    ])
else:
    print('run_quality_eval=False, bỏ qua eval.')

if quality_csv.exists():
    display(pd.read_csv(quality_csv))

## 13. ONNX export/parity cho C1 (tuỳ chọn)

Mặc định tắt. Bật `CONFIG['run_onnx_export'] = True` khi cần kiểm tra deploy streaming.

In [ ]:
onnx_csv = Path(CONFIG['results_dir']) / 'ablation_onnx.csv'
onnx_dir = Path(CONFIG['onnx_dir'])

if CONFIG.get('run_onnx_export', False):
    if not best_ckpt.exists():
        raise FileNotFoundError(f'Không tìm thấy checkpoint để export ONNX: {best_ckpt}')
    run_py([
        'ablation/export_ablation_onnx.py',
        '--config-id', CONFIG['config_id'],
        '--checkpoint', best_ckpt,
        '--output-dir', onnx_dir,
        '--results', onnx_csv,
        '--device', 'cpu',
    ])
else:
    print('run_onnx_export=False, bỏ qua ONNX export.')

if onnx_csv.exists():
    display(pd.read_csv(onnx_csv))

## 14. Tổng hợp CSV và nén kết quả

Tạo `ablation_summary.csv` cho riêng C1 và zip toàn bộ workspace output để tải từ Kaggle.

In [ ]:
from IPython.display import FileLink, display

summary_csv = Path(CONFIG['results_dir']) / 'ablation_summary.csv'
run_py([
    'ablation/collect_ablation_results.py',
    '--arch', arch_csv,
    '--quality', quality_csv,
    '--onnx', onnx_csv,
    '--output', summary_csv,
])

print('\nCSV kết quả:')
for csv_path in [arch_csv, quality_csv, onnx_csv, summary_csv]:
    if csv_path.exists():
        print(f'  {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)')
        display(pd.read_csv(csv_path))

archive_base = Path('/kaggle/working/c1_ablation_training_output')
archive_path = archive_base.with_suffix('.zip')
if archive_path.exists():
    archive_path.unlink()
shutil.make_archive(str(archive_base), 'zip', root_dir=str(WORK_DIR))
print(f'Đã nén kết quả: {archive_path}')
display(FileLink(str(archive_path)))